# Exercise 8: Dispersion and the closed orbit

**Block 3: Dispersion and chromaticity**

**Learning objectives**
* Find the **closed orbit** of an off-momentum particle with the 5×5 one-turn matrix: $z = (I − M)^{-1} d \delta$.
* Compute the **dispersion function** $D(s)$ around the ring.
* See in phase space that an off-momentum particle oscillates around a shifted centre $D\delta$.

**Motivation.** Not all particles have exactly the design momentum $p_0$. A particle with a bit more
momentum is bent less by the dipoles, so it settles on a slightly different orbit, further out.
In phase space its ellipse is simply moved sideways. This shift is what we call dispersion.

> **Student version.** Fill in the cells marked `# YOUR CODE HERE` and run the check cells after them: they tell you when you are right.

## The 5×5 matrix

We add $\delta = \Delta p/p_0$ as a fifth coordinate. It does not change (no RF in our ring for now), so the
last row of the matrix is (0, 0, 0, 0, 1). The dipoles fill the last column: a particle with $\delta = 1$
picks up $x += R_{16}$ and $x' += R_{26}$ in each dipole
(for a sector dipole $R_{16} = \rho(1 − \cos\theta)$, $R_{26} = \sin\theta$).

$$\begin{pmatrix} x \\ x' \\ \delta \end{pmatrix}_{\text{after}} =
\begin{pmatrix} M_{11} & M_{12} & R_{16} \\ M_{21} & M_{22} & R_{26} \\ 0 & 0 & 1 \end{pmatrix}
\begin{pmatrix} x \\ x' \\ \delta \end{pmatrix}_{\text{before}}$$

(the vertical block sits in rows 3–4; this ring has no vertical bends, so the vertical dispersion is 0).

In [ ]:
# --- setup: make `beamlib` importable --------------------------------------
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "beamlib").is_dir())
sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt
from beamlib import elements as E, optics as O, lattices as L, tracking as T, diagnostics as D, beams as B, plotting as P
from beamlib.plotting import COLORS, MM
P.setup()

ring = L.fodo_ring()
one_turn = T.make_one_turn_map(ring)
M5 = O.one_turn_matrix(ring)
np.set_printoptions(precision=4, suppress=True)
print("one-turn matrix (x, px, y, py, delta):\n", M5)

## 1. The closed orbit

The **closed orbit** is the orbit that comes back to itself after one turn. For a particle with
momentum deviation $\delta$ it is the vector $z$ = ($x$, $p_x$, $y$, $p_y$) with

$$z = M_4\, z + d\,\delta \quad\Rightarrow\quad z = (I - M_4)^{-1}\, d\,\delta,$$

where *M*₄ = `M5[:4, :4]` and *d* = `M5[:4, 4]` is the dispersion column.

**Task.** Implement `closed_orbit(M5, delta)` with `np.linalg.solve` (better than computing the inverse).

In [ ]:
def closed_orbit(M5: np.ndarray, delta: float) -> np.ndarray:
    """Closed orbit (x, px, y, py) for momentum deviation delta."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
delta = 2e-3
co = closed_orbit(M5, delta)
print(f"closed orbit for delta = {delta}: x = {co[0] * MM:.4f} mm, px = {co[1] * MM:.2e} mrad")
assert np.allclose(co, O.closed_orbit_linear(M5, delta))

# Check by tracking: a particle launched on the closed orbit must come back to it.
# (Tracking also makes the quadrupoles a tiny bit weaker for delta > 0 (next exercise), so for an
#  exact check we use the one-turn matrix *at this delta*.)
co_exact = closed_orbit(O.one_turn_matrix(ring, delta), delta)
p = np.r_[co_exact, delta][:, None]
assert np.allclose(one_turn(p)[:4, 0], co_exact, atol=1e-15), "a closed orbit must close!"
print(f"difference between the two closed orbits: {abs(co_exact[0] - co[0]) * MM:.1e} mm (second order in delta)")
print("GOOD! Closed orbit found and verified by tracking")

The closed orbit is proportional to $\delta$. The proportionality factor is the **dispersion**
$D = x_\text{co}/\delta$ (and $D' = p_{x,\text{co}}/\delta$). Here, at the start of the ring (centre of QF):

In [ ]:
D0, Dp0 = co[0] / delta, co[1] / delta
print(f"D = {D0:.4f} m, D' = {Dp0:.2e}")

## 2. Dispersion around the ring

The vector $(D, D', 0, 0, 1)$ is transported by the 5×5 matrices like any particle.
`O.cumulative_matrices(ring)` gives the matrix from the start to many points $s$.

**Task.** Compute $D(s)$ and plot it together with $\beta_x(s)$.

In [ ]:
s, elem, Ms = O.cumulative_matrices(ring, n_slices=20)
# YOUR CODE HERE
raise NotImplementedError("Replace this line with your code")
tw = O.ring_twiss(ring, n_slices=20)
assert np.allclose(Ds, tw["dx"]) and np.isclose(Ds[-1], D0), "D must be periodic"
print("GOOD! Dispersion is periodic")

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(tw["s"], tw["betx"], color=COLORS["x"], label=r"$\beta_x$ [m]")
ax.plot(s, Ds, color=COLORS["accent"], lw=2, label=r"$D$ [m]")
ax.set_xlim(0, 12); ax.set_xlabel("s [m]"); ax.legend(); ax.set_title("first two cells")
plt.show()

$D$ is largest at the focusing quads (where $\beta_x$ is also largest) and positive everywhere: the
particles with more momentum are always outside the reference orbit.

## 3. Off-momentum particles in phase space

**Predict first.** Three particles have $\delta = −2\cdot10^{-3}$, 0 and $+2\cdot10^{-3}$ and all start with the *same*
betatron offset of 1.5 mm from their own closed orbit. Draw what you expect in the ($x$, $p_x$)
plane after 300 turns.

*Your prediction:*

In [ ]:
deltas = np.array([-2e-3, 0.0, 2e-3])
bunch = np.zeros((5, 3))
for k, d in enumerate(deltas):
    bunch[:4, k] = closed_orbit(O.one_turn_matrix(ring, d), d)
    bunch[0, k] += 1.5e-3                     # betatron offset
    bunch[4, k] = d
res = T.track(bunch, one_turn, 300)

fig, ax = plt.subplots(figsize=(7, 5))
colors = ["b", COLORS["x"], COLORS["y"]]
for k, d in enumerate(deltas):
    ax.scatter(res.x[:, k] * MM, res.px[:, k] * MM, s=3, color=colors[k], label=f"δ = {d:+.0e}")
    ax.plot(D0 * d * MM, 0, "+", color=colors[k], ms=15, mew=2)
ax.set_xlabel(r"$x$ [mm]"); ax.set_ylabel(r"$p_x$ [mrad]"); ax.legend()
ax.set_title(r"Same ellipse, shifted by $D\delta$ (crosses)")
plt.show()

**Task.** Measure the centre of each ellipse as the average of $x$ over the 300 turns, and check it equals $D\delta$.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("Replace this line with your code")
print("measured centres [mm]:", (centres * MM).round(4), "  D*delta [mm]:", (D0 * deltas * MM).round(4))
assert np.allclose(centres, D0 * deltas, rtol=0.02, atol=3e-5)   # averaging 300 turns is not perfect
print("GOOD! The ellipse centre is shifted by D * delta")

## 4. A beam with momentum spread

A real beam has a spread of momenta, with rms $\sigma_\delta. Each particle oscillates around its own
closed orbit $D\delta$, so the beam size gets two independent contributions:

$$\sigma_x^2 = \underbrace{\beta\,\varepsilon_{rms}}_{\text{betatron}} + \underbrace{D^2\sigma_\delta^2}_{\text{dispersion}} .$$

In [ ]:
N, eps, sig_d = 50_000, 1e-6, 1e-3
twx = O.twiss_from_matrix(M5[:2, :2])
bunch = B.matched_gaussian(N, eps, twx["beta"], twx["alpha"], delta_rms=sig_d, rng=1)
bunch[0, :] += D0 * bunch[4, :]                 # put every particle around its own closed orbit
bunch[1, :] += Dp0 * bunch[4, :]
sx_measured = bunch[0, :].std()
sx_formula = np.sqrt(twx["beta"] * eps + D0**2 * sig_d**2)
print(f"sigma_x measured {sx_measured * MM:.4f} mm, formula {sx_formula * MM:.4f} mm "
      f"(betatron only: {np.sqrt(twx['beta'] * eps) * MM:.3f} mm)")
assert abs(sx_measured / sx_formula - 1) < 0.01

## Reflection questions

1. Why is the dispersion largest where $\beta_x$ is largest in a FODO lattice?
2. If you wanted a beam that is as small as possible (for example at a collision point), what
   would you want $D$ to be there?
3. An off-momentum particle oscillates around $D\delta$. Does it also oscillate at the same *tune* as an
   on-momentum particle? (Look at Exercise 9.)